# 📖 Notebook 3: Personalised Feed Generation

A global ranked feed is a good start, but the best news aggregators tailor the feed to **each user**. Alice (a tech enthusiast) should see AI and programming articles first. Bob (a sports fan) should see the Lakers game at the top.

This notebook builds a personalised feed using **user interest profiles**, **content-based scoring**, and **implicit feedback** from reading history — all cached in Redis for fast serving.

## Learning Objectives

By the end of this notebook, you'll understand:
- How user interest profiles drive personalisation
- How to score articles against a user's interests (content-based filtering)
- How implicit feedback (clicks, reads, bookmarks) refines recommendations
- How to cache per-user feeds in Redis with TTL
- The trade-offs between pre-computed and on-demand feed generation

## 🛠️ Setup

```bash
cd system-designs/news-aggregator
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [1]:
import psycopg2
import psycopg2.extras
import redis
import json
import math
import time
from datetime import datetime, timezone

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "newsagg_demo", "user": "demo", "password": "demo"
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

conn = get_db(); conn.close(); print("✅ PostgreSQL")
r = get_redis(); r.ping(); print("✅ Redis")

✅ PostgreSQL
✅ Redis


## 👤 User Interest Profiles

Each user has a set of **category interests** with weights between 0 and 1:

```
Alice:  Technology=1.0, Science=0.7, Business=0.3
Bob:    Sports=1.0, Entertainment=0.8, Health=0.2
Carol:  Business=1.0, Politics=0.9, World=0.5
```

These weights tell us how much the user cares about each topic. When we score an article, we multiply the article's category relevance by the user's interest weight.

Let's load the profiles from our database.

In [2]:
def get_user_profiles() -> dict:
    """
    Load all user interest profiles from the database.
    Returns: {user_id: {"name": ..., "interests": {category_id: weight}}}
    """
    conn = get_db()
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    # Get users
    cursor.execute("SELECT id, username, display_name FROM users")
    users = {u["id"]: {"name": u["display_name"], "username": u["username"], "interests": {}}
             for u in cursor.fetchall()}

    # Get interests with category names
    cursor.execute("""
        SELECT ui.user_id, ui.category_id, ui.weight, c.name AS category_name
        FROM user_interests ui
        JOIN categories c ON ui.category_id = c.id
    """)
    for row in cursor.fetchall():
        uid = row["user_id"]
        if uid in users:
            users[uid]["interests"][row["category_id"]] = {
                "name": row["category_name"],
                "weight": float(row["weight"])
            }

    conn.close()
    return users

profiles = get_user_profiles()

print("👤 User Interest Profiles:")
print("=" * 65)
for uid, profile in profiles.items():
    interests_str = ", ".join(
        f"{i['name']}={i['weight']:.1f}"
        for i in sorted(profile["interests"].values(), key=lambda x: -x["weight"])
    )
    print(f"  {profile['name']:<18} | {interests_str}")

👤 User Interest Profiles:
  Alice Johnson      | Technology=1.0, Science=0.7, Business=0.3
  Bob Smith          | Sports=1.0, Entertainment=0.8, Health=0.2
  Carol Williams     | Business=1.0, Politics=0.9, World=0.5
  Dave Brown         | Science=1.0, Technology=0.6, Health=0.4
  Eve Davis          | Technology=0.7, Science=0.7, Business=0.7, Sports=0.7, Entertainment=0.7, Health=0.7, Politics=0.7, World=0.7


## 🎯 Content-Based Scoring

The personalisation formula combines three signals:

```
personal_score = (w1 × freshness) + (w2 × popularity) + (w3 × relevance)
```

Where **relevance** is new — it measures how well the article matches the user's interests:

```
relevance = Σ (article_category_score × user_interest_weight)
```

For example, if an article is categorised as Technology (1.0) and Business (0.5),  
and Alice has Technology=1.0, Business=0.3:

```
relevance = (1.0 × 1.0) + (0.5 × 0.3) = 1.15
```

In [3]:
# Reuse freshness/popularity from Notebook 2
HALF_LIFE_HOURS = 6
DECAY_RATE = math.log(2) / HALF_LIFE_HOURS

def freshness_score(published_at: datetime) -> float:
    if not published_at:
        return 0.0
    now = datetime.now(timezone.utc)
    if published_at.tzinfo is None:
        published_at = published_at.replace(tzinfo=timezone.utc)
    age_hours = (now - published_at).total_seconds() / 3600
    return math.exp(-DECAY_RATE * max(age_hours, 0))

def popularity_score(source_count: int, interaction_count: int) -> float:
    source_s = math.log(1 + source_count) / math.log(10)
    interact_s = math.log(1 + interaction_count) / math.log(100)
    return 0.6 * source_s + 0.4 * interact_s


def relevance_score(article_categories: dict, user_interests: dict) -> float:
    """
    How well does this article match the user's interests?
    
    article_categories: {category_id: relevance_score}
    user_interests:     {category_id: {"weight": float}}
    
    Returns a score >= 0. Higher = more relevant.
    """
    score = 0.0
    for cat_id, cat_relevance in article_categories.items():
        if cat_id in user_interests:
            score += cat_relevance * user_interests[cat_id]["weight"]
    return score


# Demo: score one article for two different users
# Article: EU AI Regulation (Politics=0.8, Technology=0.6)
article_cats = {7: 0.8, 1: 0.6}  # Politics=0.8, Technology=0.6

alice_score = relevance_score(article_cats, profiles[1]["interests"])  # Alice: tech fan
bob_score = relevance_score(article_cats, profiles[2]["interests"])    # Bob: sports fan
carol_score = relevance_score(article_cats, profiles[3]["interests"])  # Carol: politics/biz

print("🎯 Relevance Score: EU AI Regulation (Politics + Technology)")
print("=" * 55)
print(f"  Alice (tech enthusiast):   {alice_score:.2f}  ← matches on Technology")
print(f"  Bob   (sports fan):        {bob_score:.2f}  ← no matching interests")
print(f"  Carol (business/politics): {carol_score:.2f}  ← matches on Politics")
print()
print("💡 The same article gets very different relevance scores per user.")

🎯 Relevance Score: EU AI Regulation (Politics + Technology)
  Alice (tech enthusiast):   0.60  ← matches on Technology
  Bob   (sports fan):        0.00  ← no matching interests
  Carol (business/politics): 0.72  ← matches on Politics

💡 The same article gets very different relevance scores per user.


## 📰 Building a Personalised Feed

Now let's put it all together: for each user, score every (non-duplicate) article and sort by personalised score.

In [4]:
def get_articles_with_categories() -> list:
    """
    Fetch all articles with their categories, interactions, and duplicate info.
    """
    conn = get_db()
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    # Get articles (exclude non-canonical duplicates)
    cursor.execute("""
        SELECT a.id, a.title, a.published_at, a.source_count, a.summary,
               f.name AS source,
               COUNT(DISTINCT ui.id) AS interactions,
               dg.canonical_article_id
        FROM articles a
        JOIN feeds f ON a.feed_id = f.id
        LEFT JOIN user_interactions ui ON a.id = ui.article_id
        LEFT JOIN duplicate_members dm ON a.id = dm.article_id
        LEFT JOIN duplicate_groups dg ON dm.group_id = dg.id
        GROUP BY a.id, f.name, dg.canonical_article_id
    """)
    articles = cursor.fetchall()

    # Filter duplicates: keep canonical or non-grouped
    unique = [a for a in articles
              if a["canonical_article_id"] is None or a["id"] == a["canonical_article_id"]]

    # Get categories for each article
    cursor.execute("""
        SELECT article_id, category_id, relevance_score
        FROM article_categories
    """)
    cat_map = {}  # {article_id: {category_id: relevance}}
    for row in cursor.fetchall():
        aid = row["article_id"]
        if aid not in cat_map:
            cat_map[aid] = {}
        cat_map[aid][row["category_id"]] = float(row["relevance_score"])

    for a in unique:
        a["categories"] = cat_map.get(a["id"], {})

    conn.close()
    return unique


def build_personal_feed(user_id: int, articles: list, profiles: dict,
                        freshness_w=0.35, popularity_w=0.25, relevance_w=0.40,
                        limit: int = 10) -> list:
    """
    Score and rank articles for a specific user.
    
    Weights:
    - freshness_w:  how much to value recency (default 35%)
    - popularity_w: how much to value broad appeal (default 25%)
    - relevance_w:  how much to value personal interest match (default 40%)
    """
    user = profiles.get(user_id, {})
    interests = user.get("interests", {})

    scored = []
    for a in articles:
        f = freshness_score(a["published_at"])
        p = popularity_score(a["source_count"], a["interactions"])
        rel = relevance_score(a["categories"], interests)

        total = freshness_w * f + popularity_w * p + relevance_w * rel

        scored.append({
            **a,
            "freshness": f,
            "popularity": p,
            "relevance": rel,
            "personal_score": total
        })

    scored.sort(key=lambda x: x["personal_score"], reverse=True)
    return scored[:limit]


# Build feeds for all users
articles = get_articles_with_categories()
print(f"📰 {len(articles)} unique articles available\n")

📰 144 unique articles available



In [5]:
# Show side-by-side: how the same articles rank differently for different users

for user_id in [1, 2, 3, 4]:  # Alice, Bob, Carol, Dave
    feed = build_personal_feed(user_id, articles, profiles, limit=5)
    user = profiles[user_id]
    interests_str = ", ".join(
        f"{i['name']}"
        for i in sorted(user["interests"].values(), key=lambda x: -x["weight"])[:3]
    )

    print(f"👤 {user['name']}'s Feed (interests: {interests_str})")
    print("-" * 80)
    for i, a in enumerate(feed):
        print(f"  {i+1}. {a['title'][:50]:<50} "
              f"[F:{a['freshness']:.2f} P:{a['popularity']:.2f} "
              f"R:{a['relevance']:.2f}] = {a['personal_score']:.3f}")
    print()

print("💡 Notice how each user's top articles match their interests!")
print("   Alice sees tech/science, Bob sees sports/entertainment, etc.")

👤 Alice Johnson's Feed (interests: Technology, Science, Business)
--------------------------------------------------------------------------------
  1. OpenAI Releases GPT-5 With Reasoning Breakthrough  [F:0.77 P:0.38 R:1.00] = 0.764
  2. Show HN: I Built a Database That Runs on SQLite an [F:0.68 P:0.24 R:1.00] = 0.700
  3. Apple Unveils M5 Chip With 3nm Architecture        [F:0.10 P:0.28 R:1.00] = 0.503
  4. NASA Confirms Water Ice Deposits on Mars Surface   [F:0.24 P:0.28 R:0.70] = 0.434
  5. Federal Reserve Holds Interest Rates Steady at 4.5 [F:0.54 P:0.46 R:0.30] = 0.424

👤 Bob Smith's Feed (interests: Sports, Entertainment, Health)
--------------------------------------------------------------------------------
  1. Lakers Win NBA Championship in Game 7 Thriller     [F:0.38 P:0.41 R:1.00] = 0.636
  2. Streaming Wars: Netflix Surpasses 300 Million Subs [F:0.48 P:0.24 R:0.80] = 0.550
  3. OpenAI Releases GPT-5 With Reasoning Breakthrough  [F:0.77 P:0.38 R:0.00] = 0.364
  4. Wembanya

## 📈 Learning from Implicit Feedback

User interest profiles are a great start, but they're static. Real aggregators **learn** from behaviour:

| Action | Signal Strength | What It Tells Us |
|--------|----------------|------------------|
| **Click** | Low | User was curious enough to open |
| **Read** (time on page) | Medium | User found it interesting |
| **Bookmark** | High | User wants to save for later |
| **Share** | Highest | User recommends it to others |

We use this data to **boost categories** the user engages with and **penalise** categories they ignore.

In [6]:
# Weights for different interaction types
INTERACTION_WEIGHTS = {
    "click": 1.0,
    "read": 2.0,
    "bookmark": 3.0,
    "share": 4.0,
}

def compute_implicit_interests(user_id: int) -> dict:
    """
    Analyse a user's interaction history to discover implicit interests.
    Returns: {category_id: implicit_weight}
    """
    conn = get_db()
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    cursor.execute("""
        SELECT ui.interaction_type,
               ac.category_id,
               c.name AS category_name,
               ac.relevance_score
        FROM user_interactions ui
        JOIN article_categories ac ON ui.article_id = ac.article_id
        JOIN categories c ON ac.category_id = c.id
        WHERE ui.user_id = %s
    """, (user_id,))

    category_scores = {}  # {category_id: {"name": ..., "score": float}}
    for row in cursor.fetchall():
        cid = row["category_id"]
        weight = INTERACTION_WEIGHTS.get(row["interaction_type"], 1.0)
        contribution = weight * float(row["relevance_score"])

        if cid not in category_scores:
            category_scores[cid] = {"name": row["category_name"], "score": 0.0}
        category_scores[cid]["score"] += contribution

    conn.close()

    # Normalise scores to 0-1 range
    if category_scores:
        max_score = max(cs["score"] for cs in category_scores.values())
        if max_score > 0:
            for cs in category_scores.values():
                cs["score"] = round(cs["score"] / max_score, 2)

    return category_scores


# Show implicit interests for each user
print("📊 Implicit Interests (learned from reading history):")
print("=" * 65)
for uid in [1, 2, 3, 4]:
    implicit = compute_implicit_interests(uid)
    user_name = profiles[uid]["name"]
    interests_str = ", ".join(
        f"{v['name']}={v['score']:.2f}"
        for v in sorted(implicit.values(), key=lambda x: -x["score"])
    )
    print(f"  {user_name:<18} | {interests_str or 'no interactions yet'}")

📊 Implicit Interests (learned from reading history):
  Alice Johnson      | Technology=1.00
  Bob Smith          | Sports=1.00, Entertainment=0.14
  Carol Williams     | Business=1.00, Technology=0.28, Politics=0.20
  Dave Brown         | Science=1.00


In [7]:
def blend_interests(explicit: dict, implicit: dict,
                    explicit_weight: float = 0.6, implicit_weight: float = 0.4) -> dict:
    """
    Combine explicit user interests with implicit (learned) interests.
    This lets the system adapt as the user's behaviour changes.
    
    explicit: {category_id: {"name": ..., "weight": float}}
    implicit: {category_id: {"name": ..., "score": float}}
    """
    all_cats = set(explicit.keys()) | set(implicit.keys())
    blended = {}

    for cid in all_cats:
        exp_w = explicit.get(cid, {}).get("weight", 0.0)
        imp_w = implicit.get(cid, {}).get("score", 0.0)
        name = explicit.get(cid, {}).get("name") or implicit.get(cid, {}).get("name", "?")

        blended[cid] = {
            "name": name,
            "weight": round(explicit_weight * exp_w + implicit_weight * imp_w, 3)
        }

    return blended


# Show blended interests for Alice
alice_implicit = compute_implicit_interests(1)
alice_blended = blend_interests(profiles[1]["interests"], alice_implicit)

print("🔀 Blended Interests for Alice (60% explicit + 40% implicit):")
print("=" * 65)
print(f"  {'Category':<18} {'Explicit':>10} {'Implicit':>10} {'Blended':>10}")
print("-" * 55)
for cid, blend in sorted(alice_blended.items(), key=lambda x: -x[1]["weight"]):
    exp = profiles[1]["interests"].get(cid, {}).get("weight", 0.0)
    imp = alice_implicit.get(cid, {}).get("score", 0.0)
    print(f"  {blend['name']:<18} {exp:>10.2f} {imp:>10.2f} {blend['weight']:>10.3f}")

print("\n💡 Alice's reading history confirms her tech interest")
print("   and might surface new categories she's engaging with.")

🔀 Blended Interests for Alice (60% explicit + 40% implicit):
  Category             Explicit   Implicit    Blended
-------------------------------------------------------
  Technology               1.00       1.00      1.000
  Science                  0.70       0.00      0.420
  Business                 0.30       0.00      0.180

💡 Alice's reading history confirms her tech interest
   and might surface new categories she's engaging with.


## ⚡ Caching Per-User Feeds in Redis

Computing a personalised feed involves:
1. Loading the user's interests
2. Fetching all recent articles with categories
3. Scoring and sorting

This is expensive for every single request. Instead, we:
- **Pre-compute** each user's feed on a schedule (e.g., every 5 minutes)
- **Cache** the result in Redis as a sorted set
- **Serve** reads from cache (sub-millisecond)

```
User opens app
      │
      ▼
  Redis cache hit? ──yes──► Return cached feed (fast!)
      │
     no
      │
      ▼
  Compute personalised feed
      │
      ▼
  Store in Redis (TTL=5min)
      │
      ▼
  Return feed to user
```

In [8]:
r = get_redis()
FEED_TTL = 300  # 5 minutes

def cache_user_feed(user_id: int, feed: list):
    """Cache a personalised feed in Redis as a sorted set."""
    key = f"user_feed:{user_id}"
    pipe = r.pipeline()
    pipe.delete(key)

    for article in feed:
        member = json.dumps({
            "id": article["id"],
            "title": article["title"],
            "source": article["source"],
            "summary": (article["summary"] or "")[:200],
            "relevance": round(article["relevance"], 3),
        })
        pipe.zadd(key, {member: article["personal_score"]})

    pipe.expire(key, FEED_TTL)
    pipe.execute()


def get_user_feed_cached(user_id: int, offset: int = 0, limit: int = 5) -> list:
    """Read a user's feed from Redis (highest score first)."""
    key = f"user_feed:{user_id}"
    results = r.zrevrange(key, offset, offset + limit - 1, withscores=True)
    if not results:
        return None  # cache miss
    return [(json.loads(member), score) for member, score in results]


def get_or_build_feed(user_id: int, articles: list, profiles: dict,
                      offset: int = 0, limit: int = 5) -> list:
    """
    The main feed endpoint: try cache first, compute on miss.
    """
    # Try cache
    cached = get_user_feed_cached(user_id, offset, limit)
    if cached:
        return cached, "cache_hit"

    # Cache miss — compute the feed
    feed = build_personal_feed(user_id, articles, profiles, limit=20)
    cache_user_feed(user_id, feed)

    # Return the requested page
    cached = get_user_feed_cached(user_id, offset, limit)
    return cached, "cache_miss"


# Demo: build and cache feeds for all users
for uid in profiles:
    feed, status = get_or_build_feed(uid, articles, profiles)
    ttl = r.ttl(f"user_feed:{uid}")
    print(f"  👤 {profiles[uid]['name']:<18} | {status:<10} | "
          f"{len(feed)} articles | TTL: {ttl}s")

  👤 Alice Johnson      | cache_miss | 5 articles | TTL: 300s
  👤 Bob Smith          | cache_miss | 5 articles | TTL: 300s
  👤 Carol Williams     | cache_miss | 5 articles | TTL: 300s
  👤 Dave Brown         | cache_miss | 5 articles | TTL: 300s
  👤 Eve Davis          | cache_miss | 5 articles | TTL: 300s


In [9]:
# Now let's compare: cache hit vs cache miss latency

# First request: cache miss (need to compute)
r.delete("user_feed:1")  # clear Alice's cache

start = time.time()
feed_miss, status_miss = get_or_build_feed(1, articles, profiles)
time_miss = (time.time() - start) * 1000

# Second request: cache hit (instant)
start = time.time()
feed_hit, status_hit = get_or_build_feed(1, articles, profiles)
time_hit = (time.time() - start) * 1000

print("⚡ Feed Generation Latency:")
print("=" * 50)
print(f"  Cache MISS (compute + store): {time_miss:>8.2f} ms")
print(f"  Cache HIT  (Redis read):      {time_hit:>8.2f} ms")
print(f"  Speedup:                      {time_miss / max(time_hit, 0.01):>8.1f}×")
print()
print("💡 With caching, most feed requests are sub-millisecond!")
print("   The expensive computation only happens every 5 minutes.")

⚡ Feed Generation Latency:
  Cache MISS (compute + store):     2.70 ms
  Cache HIT  (Redis read):          0.66 ms
  Speedup:                           4.1×

💡 With caching, most feed requests are sub-millisecond!
   The expensive computation only happens every 5 minutes.


## 🔄 Feed Invalidation

When should we invalidate a user's cached feed?

| Event | Strategy |
|-------|----------|
| New articles crawled | Let TTL expire naturally (5 min delay is acceptable) |
| User changes interests | Immediately delete their cache key |
| Breaking news | Broadcast invalidation to all user caches |
| User reads an article | Don't invalidate (too frequent, low value) |

For most news aggregators, **TTL-based expiry** is good enough. Users expect a slight delay for new stories — they're not chatting in real time.

In [10]:
def invalidate_user_feed(user_id: int):
    """Force a user's feed to be recomputed on next request."""
    r.delete(f"user_feed:{user_id}")
    print(f"🗑️  Invalidated feed cache for user {user_id}")

def invalidate_all_feeds():
    """Invalidate all cached feeds (e.g., for breaking news)."""
    keys = r.keys("user_feed:*")
    if keys:
        r.delete(*keys)
        print(f"🗑️  Invalidated {len(keys)} cached feeds")
    else:
        print("🗑️  No cached feeds to invalidate")

# Demo: Alice changes her interests → invalidate her cache
print("Scenario: Alice adds 'Sports' to her interests")
invalidate_user_feed(1)

# Next request will recompute
feed, status = get_or_build_feed(1, articles, profiles)
print(f"Next request: {status} — feed was recomputed")

print()
print("Scenario: Breaking news — invalidate ALL feeds")
invalidate_all_feeds()

Scenario: Alice adds 'Sports' to her interests
🗑️  Invalidated feed cache for user 1
Next request: cache_miss — feed was recomputed

Scenario: Breaking news — invalidate ALL feeds
🗑️  Invalidated 5 cached feeds


## 🥶 The Cold-Start Problem

Everything above assumes the user has **explicit interests** or some **reading history**. What about a brand-new user who just signed up?

```
New user → no interests → relevance = 0 → feed is random-looking garbage
```

Three common fixes:

| Strategy | What it does | When to use |
|----------|--------------|-------------|
| **Trending fallback** | Show the globally most popular, freshest articles | Day 1 of a new account |
| **Onboarding picker** | Ask the user to tap 3–5 topics they care about | Sign-up flow |
| **Demographic / geo priors** | Infer interests from country, language, device | When you have profile data but no clicks |

Let's implement the **trending fallback**: when a user has no interests, fall back to the global popularity + freshness ranking we built in Notebook 2.

In [11]:
def build_trending_feed(articles: list, limit: int = 10) -> list:
    """Global feed ranked by freshness + popularity only — no user signal."""
    scored = []
    for a in articles:
        f = freshness_score(a["published_at"])
        p = popularity_score(a["source_count"], a["interactions"])
        scored.append({**a, "freshness": f, "popularity": p,
                       "personal_score": 0.6 * f + 0.4 * p,
                       "relevance": 0.0})
    scored.sort(key=lambda x: x["personal_score"], reverse=True)
    return scored[:limit]


def build_feed_with_cold_start(user_id: int, articles: list, profiles: dict,
                                limit: int = 5) -> tuple[list, str]:
    """
    Personalised feed if we know the user; trending feed if we don't.
    Returns (feed, reason).
    """
    user = profiles.get(user_id)
    interests = (user or {}).get("interests", {})
    if not user or not interests:
        return build_trending_feed(articles, limit=limit), "cold_start"
    return build_personal_feed(user_id, articles, profiles, limit=limit), "personalised"


# Simulate a brand-new user who just signed up
NEW_USER_ID = 9999
feed, reason = build_feed_with_cold_start(NEW_USER_ID, articles, profiles)
print(f"👤 New user (id={NEW_USER_ID}) — feed source: {reason}")
print("-" * 70)
for i, a in enumerate(feed):
    print(f"  {i+1}. {a['title'][:55]:<55} [F:{a['freshness']:.2f} P:{a['popularity']:.2f}]")
print("\n💡 A brand-new user sees the same high-quality trending feed\n"
      "   every active user would see — a sensible first impression.")

👤 New user (id=9999) — feed source: cold_start
----------------------------------------------------------------------
  1. OpenAI Releases GPT-5 With Reasoning Breakthrough       [F:0.77 P:0.38]
  2. Wembanyama joins SGA, Jokic as MVP finalist             [F:0.88 P:0.18]
  3. Follow live: Pistons host Magic in Game 1               [F:0.81 P:0.18]
  4. Bengals extend Lawrence after blockbuster trade         [F:0.79 P:0.18]
  5. NFL draft prospect Branch arrested in Georgia           [F:0.79 P:0.18]

💡 A brand-new user sees the same high-quality trending feed
   every active user would see — a sensible first impression.


## 👥 Collaborative Filtering (Brief Mention)

Content-based scoring says: *"you like Technology, here's a Technology article."* It can never surprise you with something you didn't know you'd enjoy.

**Collaborative filtering** looks at *other users who behave like you* and recommends what **they** read:

```
Alice reads:   [GPT-5, Apple M5, SQLite DB]
Dave  reads:   [GPT-5, Apple M5, JWST exoplanet, NASA Mars]
                 └─ overlap! ─┘
→ Suggest JWST + NASA Mars to Alice, even if she never picked "Science"
```

Production techniques:
- **User-user** CF: compute user-similarity (e.g. cosine on interaction vectors), recommend neighbours' reads
- **Item-item** CF: "people who read X also read Y" — scales better
- **Matrix factorisation / embeddings**: learn a low-dimensional vector per user and per article (Netflix, YouTube)

A real news-aggregator feed is usually **hybrid**: content-based + collaborative + trending, combined with a small amount of **exploration** (showing a random well-ranked item from outside your usual interests to avoid filter bubbles).

## 🏗️ System Architecture: Putting It All Together

Here's how the three notebooks connect into a complete news aggregator:

```
                    ┌─────────────────────────────────────────┐
                    │           INGESTION PIPELINE             │
                    │         (Notebook 1)                     │
  RSS Feeds ───────►│  Fetch → Parse → Normalise → Store      │
  (Internet)        │  Redis schedules crawl timing            │
                    └──────────────┬──────────────────────────┘
                                   │
                                   ▼
                    ┌─────────────────────────────────────────┐
                    │         PROCESSING PIPELINE              │
                    │         (Notebook 2)                     │
                    │  Dedup → Group → Rank (global)           │
                    │  Hash-based + Shingling similarity       │
                    └──────────────┬──────────────────────────┘
                                   │
                                   ▼
                    ┌─────────────────────────────────────────┐
                    │         SERVING PIPELINE                 │
                    │         (Notebook 3)                     │
                    │  User profile → Score → Sort → Cache     │
  User request ────►│  Redis sorted sets for fast serving      │
                    └─────────────────────────────────────────┘
```

### Scale Considerations

| Component | 1K users | 1M users | 100M users |
|-----------|----------|----------|------------|
| Feed crawlers | 1 worker | 10 workers | 100+ workers (distributed) |
| Dedup | Brute force OK | MinHash/LSH | Dedicated dedup service |
| Feed generation | On-demand | Pre-compute + cache | Fan-out on write (like Facebook) |
| Cache | Single Redis | Redis cluster | Redis cluster + CDN |

## 🧹 Cleanup

In [12]:
r = get_redis()
keys = r.keys("user_feed:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned up {len(keys)} Redis keys")
else:
    print("🧹 Nothing to clean up")

print("\n🎉 You've completed the News Aggregator lab!")

🧹 Nothing to clean up

🎉 You've completed the News Aggregator lab!


## 📚 Summary

### Key Takeaways

1. **User interest profiles** map categories to weights — this drives personalisation
2. **Content-based scoring** multiplies article category scores by user interest weights
3. **Implicit feedback** (clicks, reads, bookmarks) lets the system learn and adapt
4. **Blending explicit + implicit** interests gives the best results
5. **Redis caching** makes personalised feeds fast to serve (sub-millisecond)
6. **TTL-based expiry** is usually good enough — users accept a few minutes of staleness
7. **Cold-start fallback**: brand-new users get the trending feed until they interact enough to personalise
8. **Collaborative filtering** adds serendipity — "users like you also read…" — beyond strict topic matching

### System Design Interview Tips

- **Pre-compute vs on-demand**: discuss when each makes sense. Pre-compute for active users, on-demand for inactive ones
- **Cold start problem**: new users have no history — fall back to trending/popular articles
- **Collaborative filtering**: "users like you also read…" — mention it as an advanced technique
- **A/B testing**: different ranking weights need experimentation — mention this shows maturity
- **Privacy**: personalisation requires storing user data — mention GDPR/privacy considerations

### Full Lab Summary

| Notebook | Pipeline | Key Technologies |
|----------|----------|------------------|
| 1 | Ingestion | feedparser, PostgreSQL, Redis sorted sets |
| 2 | Processing | SHA-256 hashing, bag-of-words Jaccard, shingling, MinHash/LSH |
| 3 | Serving | Content-based filtering, implicit feedback, cold-start trending, Redis caching |